# Demo 04 - a third device gets onboarded

Also the outside world, not the pipeline. This writes a file into the landing zone, the same way an
export from a newly installed device would arrive.

When a device is added it usually has history sitting on it already, so that history is loaded once
from an export while the device joins the live stream going forward. Backfill by file, then stream.

Run it twice, with the `part` widget:

- **part 1**, the device is installed and starts reporting the usual metrics
- **part 2**, its firmware is updated a few days later and it starts reporting two more:
  `psu_temp_c` and `fan_rpm`

Part 2 is the schema evolution moment. Nothing about it is contrived, a newer device reporting more
than the older ones is the normal state of any fleet that was not bought all at once.

The values are derived from the real readings of an existing device with a small offset, so the new
box has plausible temperatures and load instead of numbers from nowhere.

In [0]:
from pyspark.sql import functions as F

dbutils.widgets.text("login", "")
dbutils.widgets.text("target_catalog", "")
dbutils.widgets.text("new_device_id", "NAS9AB123")
dbutils.widgets.text("shift_days", "30")
dbutils.widgets.dropdown("part", "1", ["1", "2"])

login      = dbutils.widgets.get("login")
catalog    = dbutils.widgets.get("target_catalog")
new_id     = dbutils.widgets.get("new_device_id")
shift_days = int(dbutils.widgets.get("shift_days"))
part       = dbutils.widgets.get("part")
assert all([login, catalog, new_id])

demo    = f"{login}_demo_bronze"
landing = f"/Volumes/{catalog}/{demo}/demo_landing/readings"
source  = f"{catalog}.{demo}.sensor_readings_bronze"

print(f"part {part} -> {landing}")

In [0]:
# take one existing device as the template, keep the shape, shift the values a little
template = (spark.table(source)
            .filter(F.col("source_system") == "eventhub")
            .filter(F.col("device_id").isNotNull())
            .filter(F.col("device_id") == F.lit("NAS6CC246")))

half = template.count() // 2
assert half > 0, "run demo_02 and demo_03 first, the template comes from the ingested readings"

# part 1 takes the older half of the history, part 2 the newer one
window = (template.orderBy("reading_ts").limit(half) if part == "1"
          else template.orderBy(F.col("reading_ts").desc()).limit(half))

readings = (window
            .withColumn("device_id", F.lit(new_id))
            # the box was installed later, so its history sits in its own window.
            # without this it would be a literal duplicate of the template on the same
            # timestamps, and the two devices would draw one line on every chart
            .withColumn("reading_ts",
                        (F.to_timestamp("reading_ts")
                         + F.expr(f"INTERVAL {shift_days} DAYS")).cast("string"))
            .withColumn("cpu_usage_pct", F.round(F.col("cpu_usage_pct") * 0.95, 1))
            .withColumn("disk_temp_c",   F.col("disk_temp_c") - F.lit(2))
            .withColumn("disk_used_pct", F.round(F.col("disk_used_pct") * 0.6, 1))
            .select("device_id", "reading_ts", "cpu_usage_pct", "ram_usage_pct",
                    "disk_temp_c", "disk_used_pct", "system_health", "net_in_mbps"))

if part == "2":
    # firmware update: two metrics the older boxes never reported
    readings = (readings
                .withColumn("psu_temp_c", F.round(F.lit(34) + F.rand(7) * 8, 1))
                .withColumn("fan_rpm",    F.round(F.lit(2100) + F.rand(7) * 900).cast("int")))

print(readings.count(), "readings,", len(readings.columns), "columns")
print("window:", readings.agg(F.min("reading_ts"), F.max("reading_ts")).first())
display(readings.limit(5))

In [0]:
out = f"{landing}/{new_id}_part{part}"

readings.coalesce(1).write.mode("overwrite").format("json").save(out)

files = [f for f in dbutils.fs.ls(out) if f.name.endswith(".json")]
print(f"{out} -> {len(files)} file(s), {sum(f.size for f in files) / 1024:.0f} KB")
display(dbutils.fs.ls(landing))

The file is in the landing zone now. Nothing has read it yet.

Run the job. On part 1 the new device simply shows up, no code changed anywhere. On part 2 the
ingestion task stops on the two unknown columns, writes the wider schema, and the retry goes
through.